# Discount curves (OIS / risk-free discounting)

Deep-dive reference: **discount factor curves** for present value and curve-implied rates.

## Concept

A **discount curve** maps time (in year fractions from a base date) to **discount factors** \(DF(t)\): the present value of one unit of currency paid at \(t\). From \(DF\) you recover **zero rates** and **implied forward rates** between two times. OIS curves are the usual risk-free discounting choice for many instruments in a single currency.

## API walkthrough

`DiscountCurve` is built from `(time_years, discount_factor)` knots. Use `df(t)` for the discount factor, `zero(t)` for the continuously compounded zero rate, and `forward_rate(t1, t2)` for the forward between two times.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import DiscountCurve

curve = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
)
print("curve:", curve)
print("df(1.0) =", curve.df(1.0))
print("zero(1.0) =", curve.zero(1.0))

In [ ]:
tenors = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
print("Tenor (y)\tDF(t)\t\tzero(t)")
for t in tenors:
    print(f"{t}\t\t{curve.df(t):.6f}\t{curve.zero(t):.6f}")

t1, t2 = 1.0, 5.0
fwd = curve.forward(t1, t2)
print()
print(f"Continuous forward {t1}y -> {t2}y: {fwd:.6f}")

usd = curve
eur = DiscountCurve(
    "EUR-ESTR",
    date(2024, 1, 1),
    [(0.0, 1.0), (0.5, 0.978), (1.0, 0.955), (2.0, 0.91), (5.0, 0.78), (10.0, 0.55)],
    day_count="act_365f",
)
print()
print("Side-by-side at 5Y: USD-OIS df =", usd.df(5.0), "| EUR-ESTR df =", eur.df(5.0))
print("Side-by-side at 5Y: USD zero =", usd.zero(5.0), "| EUR zero =", eur.zero(5.0))

Beyond the last knot (\(t=10\)y here), discount factors and zeros follow the curve’s **extrapolation** rule (often **flat forward** by default). Querying **\(t=15\)** and **\(t=20\)** years shows that behavior explicitly.

In [ ]:
for t in (15.0, 20.0):
    print(f"t={t}y  df(t)={curve.df(t):.8f}  zero(t)={curve.zero(t):.6f}")

## Practical example

Discount a single future cashflow: \(PV = C \cdot DF(T)\).

In [ ]:
cashflow = 1_000_000.0
maturity_years = 3.0
df_t = curve.df(maturity_years)
pv = cashflow * df_t
print(f"Cashflow USD {cashflow:,.0f} paid in {maturity_years}y")
print(f"DF({maturity_years}) = {df_t:.6f}")
print(f"Present value = USD {pv:,.2f}")

## Takeaways

- **Knots** are \((t, DF)\); interpolation and extrapolation follow the curve builder defaults (`monotone_convex` / `flat_forward` unless you override).
- **`df` / `zero` / `forward_rate`** are the three core queries for PV, par-rate intuition, and hedging/funding views.
- **Multi-currency** workflows use one discount curve per currency (e.g. USD-OIS vs EUR-ESTR); cross-currency needs FX and often a basis, which is outside this single-curve reference.

## Calibration from market quotes

In practice, discount curves are **bootstrapped** from observable market instruments (deposit rates and swap par rates) rather than constructed from hand-picked knots.  The calibration engine accepts a JSON plan specifying quote data and curve parameters, then solves for the discount factors that reprice each instrument exactly.

The quote format uses:
- `"class": "rates"` with `"type": "deposit"` or `"type": "swap"`
- `"pillar"` as `{"tenor": {"count": N, "unit": "months"|"years"}}` or `{"date": "YYYY-MM-DD"}`

In [ ]:
import sys
sys.path.insert(0, "../..")

from _shared import REPOSITORY_ROOT
from finstack_quant.calibration import calibrate

envelope_json = (
    REPOSITORY_ROOT / "finstack-quant/calibration/examples/market_bootstrap/01_usd_discount.json"
).read_text()

result = calibrate(envelope_json)
print("Success:", result.success)
print("Iterations:", result.iterations)
print("Max residual:", f"{result.report.max_residual:.2e}")
print()

cal_curve = result.market.get_discount("USD-OIS")
print("Calibrated curve term structure:")
for t in [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]:
    print(f"  t={t:5.2f}y  DF={cal_curve.df(t):.8f}  zero={cal_curve.zero(t):.6f}")


In [ ]:
# Inspect the per-step calibration report
print(result.to_dataframe().to_string(index=False))

## Analyst program: compounding and the first bond PV

Equal numerical rate quotes under different compounding conventions imply different discount factors. Conversely, the curve's zero and forward rates must reconstruct its discount factors. The manual bond PV sums only future payments and is compared with the native instrument pricer.

In [ ]:
import json, math
from datetime import date
from finstack_quant.core.dates import DayCount
from finstack_quant.valuations.instruments import price_instrument, instrument_cashflows_json
from _shared.analyst_book import AS_OF, build_market, instruments

r, years, notional = 0.05, 2.0, 100_000_000.0
discounts = {'simple': 1 / (1 + r * years), 'annual': (1 + r) ** -years,
             'semiannual': (1 + r / 2) ** (-2 * years), 'continuous': math.exp(-r * years)}
assert discounts['simple'] > discounts['annual'] > discounts['semiannual'] > discounts['continuous']
market = build_market('foundations')
curve = market.get_discount('USD-OIS')
assert abs(math.exp(-curve.zero(5.0) * 5.0) - curve.df(5.0)) < 1e-12
assert abs(curve.df(2.0) * math.exp(-curve.forward(2.0, 5.0) * 3.0) - curve.df(5.0)) < 1e-12
bond = json.dumps(instruments('foundations')['USD-CORP'])
flows = json.loads(instrument_cashflows_json(bond, market, AS_OF, 'discounting'))
manual_pv = sum(flow['amount'] * curve.df(DayCount.ACT_365F.year_fraction(AS_OF, date.fromisoformat(flow['date'])))
                for flow in flows['flows'] if date.fromisoformat(flow['date']) > AS_OF)
native_pv = price_instrument(bond, market, AS_OF).price
assert abs(manual_pv - native_pv) < 0.01
assert flows['reconciles_with_base_value']
print({'convention_pvs': {basis: notional * df for basis, df in discounts.items()}, 'manual_bond_pv': manual_pv})

## Analyst program: an overnight compounded coupon

The native cashflow builder resolves the overnight rate specification. The elementary product is displayed beside it so that a term quote is not mistaken for an overnight period return. The demonstration uses a constant synthetic SOFR forward and a single weekly period.

In [ ]:
import json, math
from datetime import date
from decimal import Decimal
from finstack_quant.cashflows.builder import CashFlowSchedule, FloatingCouponSpec, FloatingRateSpec, OvernightCompoundingMethod, ScheduleParams
from finstack_quant.core.dates import DayCount
from finstack_quant.core.market_data import ForwardCurve, MarketContext
from finstack_quant.core.money import Money

from _shared.analyst_book import build_market
overnight_market = build_market('foundations')
assert overnight_market.get_series('FIXING:USD-SOFR').id == 'FIXING:USD-SOFR'
floating_rate = FloatingRateSpec(index_id='USD-SOFR', spread_bp=Decimal('0'), reset_frequency='1W',
    reset_lag_days=0, fixing_calendar_id='weekends_only', overnight_basis=DayCount.ACT_360,
    overnight_compounding=OvernightCompoundingMethod.COMPOUNDED_IN_ARREARS)
schedule = (CashFlowSchedule.builder().principal(Money(1_000_000.0, 'USD'), date(2025, 1, 6), date(2025, 1, 13))
    .floating_cf(FloatingCouponSpec(rate_spec=floating_rate,
        schedule=ScheduleParams(frequency='1W', day_count=DayCount.ACT_360, calendar_id='weekends_only')))
    .build(overnight_market))
overnight_flows = json.loads(schedule.to_json())['flows']
coupon = next(flow for flow in overnight_flows if flow['kind'] == 'float_reset')
period_return = math.prod(1 + 0.05 * days / 360 for days in (1, 1, 1, 1, 3)) - 1
assert abs(float(coupon['amount']['amount']) - 1_000_000 * period_return) < 1e-6
print({'compounded_period_return': period_return, 'coupon': coupon['amount']})
assert floating_rate.overnight_compounding == OvernightCompoundingMethod.COMPOUNDED_IN_ARREARS

## Equivalent rate representations

In [ ]:
import math
from _shared.analyst_book import build_market
curve=build_market('foundations').get_discount('USD-OIS')
t=2.0;df=curve.df(t)
equivalent={'simple':(1/df-1)/t,'annual':df**(-1/t)-1,'semiannual':2*(df**(-1/(2*t))-1),'continuous':-math.log(df)/t}
reconstructed=[1/(1+equivalent['simple']*t),(1+equivalent['annual'])**(-t),(1+equivalent['semiannual']/2)**(-2*t),math.exp(-equivalent['continuous']*t)]
assert max(abs(x-df) for x in reconstructed)<1e-12
print({'DF':df,'equivalent_decimal_rates':equivalent})


## Discount factor no-arbitrage identity

In [ ]:
import math
from _shared.analyst_book import build_market
curve=build_market('foundations').get_discount('USD-OIS')
a,b=2.0,5.0
continuous=curve.forward(a,b)
simple=(curve.df(a)/curve.df(b)-1)/(b-a)
assert abs(curve.df(a)*math.exp(-continuous*(b-a))-curve.df(b))<1e-12
assert abs(curve.df(a)/(1+simple*(b-a))-curve.df(b))<1e-12
print({'interval_years':b-a,'continuous_forward':continuous,'simple_forward':simple,'DF_end':curve.df(b)})


## A tenor-dependent rate shock

In [ ]:
import math
from finstack_quant.core.market_data import DiscountCurve
from _shared.analyst_book import AS_OF,build_market
curve=build_market('foundations').get_discount('USD-OIS');before=curve.df(5)
steep=DiscountCurve('USD-STEEP',AS_OF,[(t,curve.df(t)*math.exp(-0.001*(t/10)*t)) for t in sorted(set(curve.knots)|{5.0})],interp='log_linear')
base_pv,shock_pv=1000000*curve.df(5),1000000*steep.df(5)
assert shock_pv<base_pv and curve.df(5)==before
assert abs(shock_pv/base_pv-math.exp(-0.0005*5))<1e-12
print({'cashflow_USD':1000000,'payment_year':5,'zero_shock_bp':5,'base_PV':base_pv,'steepened_PV':shock_pv,'PnL':shock_pv-base_pv})
